### RaschPy Differential Item Functioning (DIF) worked example

Differential Item Functioning occurs when persons with the same overall trait level, but from different groups (e.g. Gender, age or first language), have different probabilities of success on a given item, which violates measurement principles and represents a threat to the validity of scores if unaddressed. `dif_test()`, which is available for `SLM`, `PCM`, `RSM` and `MFRM` (`MFRM` also has `drf_test()` for differential rater functioning), tests every item for DIF against a chosen reference group:

1. Reads in a file mapping persons to exogenous variables, stored as an attribute `self.exogenous`. This file does not need to have a mapping for every person to every variable; the test ignores persons who are unlabelled for the relevant variable.
2. Splits non-extreme persons into groups by a named column in `self.exogenous` (any number of groups; each "focal" group is compared to a single "reference" group, by default the largest).
3. Calibrates each group independently, on the items its persons responded to and with the object's `calibrate` settings (for `MFRM`, when the groups are scored by different facet elements, each group is anchored on the facet elements shared by both groups), then purifies both groups onto a common scale — genuinely DIF-affected items are excluded from the scale used to detect DIF so they do not distort the test.
4. Tests every item (not only the ones used to define the purified scale) via a per-item Wald test, likelihood-ratio test, or both, plus an omnibus test that tests whether DIF exists in any items in the data. $ p $-values are corrected for multiple comparisons across items by default (BH by default, Bonferroni also available) before the flagging rule is applied.

For PCM, RSM and MFRM, which have a threshold/category structure in addition to item location, `dif_test()` also separately tests for Differential Category Functioning (DCF) via `threshold_dif_table`. Here we make a distinction between DCF and the more standard DSF (Differential Step Functioning) approach, which tests for differences in threshold locations: instead, DCF tests for differences in category widths between adjacent thresholds ($ \tau_{k+1} - \tau_{k} $), tested category by category, reference vs. focal. This matters because two groups can show identical overall item difficulty — nothing flagged in `dif_table` — while still using the rating scale differently, e.g. one group's raters compress two middle categories together or all but skip one; that is a real, substantive difference in how the item functions, but it is invisible to any test of the item as a whole. The issue with DSF compared to DCF is that thresholds are interdependent; with a maximum score $ m $, there are $ m $ thresholds but only $ m - 1 $ degrees of freedom, due to the sum-to-zero constraint on Rasch-Andrich thresholds. For example, with a maximum score of 2, if one threshold shifts, the other must by definition shift by an equal and opposite amount; the standard error of the middle (only bounded) category is precisely twice that of each threshold (equal by definition). This means that a change in the functioning (width) of one category propagates through the rest of the thresholds, confounding the DSF test. DCF, on the other hand, avoids this by examining the appropriate independent unit, with the correct degrees of freedom: $ m - 1 $ df for $ m - 1 $ bounded categories. 

`dif_test()` also optionally adds an ETS-style DIF-magnitude `Category` column (`category=True`), following Zwick, Thayer & Mazzeo (1997): each item is classified 'A' (negligible), 'B+'/'B-' (slight to moderate), or 'C+'/'C-' (moderate to large). The sign gives the direction: '+' means the item is harder for the focal group than the reference group, '-' the converse. 'B' requires |Difference| ≥ 0.43 logits plus a significant Wald/Welch $ p $-value (`category_alpha=0.05` by default); 'C' requires |Difference| ≥ 0.64 logits plus a separate, one-sided test that the difference is significantly above the B boundary itself (P(|Difference| ≤ 0.43) < 0.05), not merely significantly nonzero. This classification is independent of the plain `Flagged` column (which uses `alpha`/`logit_threshold` directly) — it exists so results can also be reported in ETS categories. It is scoped to item-location DIF (`dif_table`) only: the 0.43/0.64 logit boundaries are calibrated in the literature for item-location DIF specifically, not for DCF category-width tests.

This notebook simulates two groups (A and B) with known, deliberately injected DIF, and checks whether `dif_test()` successfully detects it.

In [1]:
import numpy as np
import pandas as pd
from raschpy import SLM, PCM, RSM, MFRM
from raschpy.simulation import SLM_Sim, PCM_Sim, RSM_Sim
from raschpy.simulation.mfrm_sim import MFRM_Sim_Global, MFRM_Sim_Items, MFRM_Sim_Centrality

#### SLM

Simulate two groups of 300 persons each on the same 12 items. Group B has genuine DIF injected on three items: Item_1 (+1.0 logits, harder for B), Item_2 (-1.0 logits, easier for B), Item_3 (+0.5 logits). All other items are unaffected.

In [2]:
no_of_persons = 300
sim_a = SLM_Sim(no_of_items=12, no_of_persons=no_of_persons, item_range=2, seed=1)

items_b = sim_a.items.copy()
items_b['Item_1'] += 1.0
items_b['Item_2'] -= 1.0
items_b['Item_3'] += 0.5

sim_b = SLM_Sim(no_of_items=12, no_of_persons=no_of_persons, item_range=2,
                manual_items=items_b, manual_item_names=sim_a.item_names, seed=2)

resp_a, resp_b = sim_a.responses.copy(), sim_b.responses.copy()
resp_a.index = [f'A_{i}' for i in range(no_of_persons)]
resp_b.index = [f'B_{i}' for i in range(no_of_persons)]
responses = pd.concat([resp_a, resp_b])

exogenous = pd.DataFrame({'Group': ['A'] * no_of_persons + ['B'] * no_of_persons}, index=responses.index)

slm = SLM(responses, exogenous=exogenous)

Run the DIF test, comparing group B against reference group A:

In [3]:
slm.dif_test('Group',
             reference='A',
             selection_method='wald',   # purification method: 'wald' (default) | 'robust_z' | 'none'
             test='both',               # per-item flagging: 'wald' | 'lr' | 'both'
             omnibus=True,              # Andersen-style joint test, all items at once
             category=True,             # adds an ETS-style A/B/C DIF-magnitude category column
             correction='bh',           # multiple-comparison correction: 'bh' | 'bonferroni' | None
             no_of_samples=500)

round(slm.dif_table, 3)

,Group,Reference,Focal,Focal (purified),Difference,SE,t,p,p (corrected),Selected,Flagged,df,Category,LR,p_LR,p_LR (corrected),Flagged_LR
Item,,,,,,,,,,,,,,,,,
Item_1,B,0.190,1.057,1.105,0.915,0.224,4.085,0.000,0.000,False,True,529.107,C+,19.989,0.000,0.000,True
Item_2,B,0.894,-0.153,-0.104,-0.998,0.217,-4.600,0.000,0.000,False,True,537.698,C-,22.880,0.000,0.000,True
Item_3,B,-0.197,0.389,0.438,0.635,0.220,2.878,0.004,0.017,False,True,536.800,B+,11.996,0.001,0.002,True
Item_4,B,0.590,0.279,0.295,-0.295,0.220,-1.338,0.182,0.545,True,False,537.965,A,2.257,0.133,0.399,False
Item_5,B,-1.226,-1.036,-0.962,0.264,0.234,1.126,0.260,0.625,True,False,538.000,A,1.004,0.316,0.759,False
Item_6,B,-0.303,-0.382,-0.337,-0.035,0.224,-0.154,0.877,0.915,True,False,537.811,A,0.056,0.813,0.887,False
Item_7,B,0.075,0.078,0.134,0.058,0.218,0.268,0.789,0.915,True,False,537.835,A,0.076,0.783,0.887,False
Item_8,B,-0.026,-0.095,-0.049,-0.023,0.220,-0.106,0.915,0.915,True,False,537.694,A,0.007,0.935,0.935,False
Item_9,B,-0.850,-0.773,-0.710,0.140,0.220,0.637,0.524,0.915,True,False,537.998,A,0.321,0.571,0.887,False


Items 1-3, where DIF was injected, should be flagged with `Difference` estimates close to the specified 1.0 / -1.0 / 0.5 logit shifts. The remaining items should not be flagged.

In [4]:
slm.dif_omnibus_table  # is there DIF at all, across every item jointly

,LR,df,p,Flagged
B,59.502164,11,0.0,True


#### PCM

Simulate two groups on 15 4-category items (maximum score 3). Group B has the same item-location DIF pattern as before (Item_1/2/3), plus a pure category-width (DCF) shift on Item_4 only — its overall item location is left unchanged, but the width of Category 1 is increased by 0.8 logits, so it should be flagged in `threshold_dif_table` rather than `dif_table`.

In [5]:
no_of_persons = 1000
max_score_vector = [3] * 15
sim_a = PCM_Sim(no_of_items=15, no_of_persons=no_of_persons, max_score_vector=max_score_vector, item_range=2, seed=1)

items_b = sim_a.items.copy()
items_b['Item_1'] += 1.0
items_b['Item_2'] -= 1.0
items_b['Item_3'] += 0.5

# Pin thresholds to sim_a's own for every item except Item_4, which gets a
# deliberate category-width (DCF) shift with no item-location shift.
manual_thresholds = []
for item in sim_a.item_names:
    vals = sim_a.thresholds.loc[item].dropna().values.copy()
    if item == 'Item_4':
        # Isolate the shift to a single category width (w1): subtracting a
        # constant from every threshold preserves all pairwise differences, so
        # perturbing tau_1 alone and re-centering shifts w1 without touching w2
        # (unlike recomputing just the last threshold, which would couple them).
        vals[0] -= 0.8
        vals -= vals.mean()
    else:
        vals[-1] = -vals[:-1].sum()  # exact zero-sum, avoiding float residue
    manual_thresholds.append(vals)

sim_b = PCM_Sim(no_of_items=15, no_of_persons=no_of_persons, max_score_vector=max_score_vector, item_range=2,
                manual_items=items_b, manual_item_names=sim_a.item_names,
                manual_thresholds=manual_thresholds, seed=2)

resp_a, resp_b = sim_a.responses.copy(), sim_b.responses.copy()
resp_a.index = [f'A_{i}' for i in range(no_of_persons)]
resp_b.index = [f'B_{i}' for i in range(no_of_persons)]
responses = pd.concat([resp_a, resp_b])
exogenous = pd.DataFrame({'Group': ['A'] * no_of_persons + ['B'] * no_of_persons}, index=responses.index)

pcm = PCM(responses, max_score_vector=max_score_vector, exogenous=exogenous)

In [6]:
pcm.dif_test('Group',
             reference='A',
             selection_method='wald',
             test='both',
             omnibus=True,
             omnibus_scope='full',
             category=True,
             correction='bh',
             no_of_samples=300)

round(pcm.dif_table, 3)   # item-location DIF

,Group,Reference,Focal,Focal (purified),Difference,SE,t,p,p (corrected),Selected,Flagged,df,Category,LR,p_LR,p_LR (corrected),Flagged_LR
Item,,,,,,,,,,,,,,,,,
Item_1,B,-0.948,0.045,0.078,1.026,0.124,8.282,0.000,0.000,False,True,1966.723,C+,148.009,0.000,0.000,True
Item_2,B,-0.254,-1.233,-1.200,-0.946,0.087,-10.869,0.000,0.000,False,True,1937.915,C-,179.042,0.000,0.000,True
Item_3,B,0.725,1.126,1.159,0.434,0.105,4.127,0.000,0.000,False,True,1983.078,B+,39.499,0.000,0.000,True
Item_4,B,-0.613,-0.583,-0.546,0.067,0.099,0.680,0.497,0.677,True,False,1974.925,A,0.624,0.429,0.716,False
Item_5,B,-0.070,-0.241,-0.217,-0.147,0.100,-1.474,0.141,0.422,True,False,1995.611,A,3.206,0.073,0.220,False
Item_6,B,0.986,0.953,0.986,-0.001,0.083,-0.007,0.994,0.994,True,False,1993.386,A,0.002,0.968,0.968,False
Item_7,B,-0.284,-0.181,-0.135,0.149,0.089,1.680,0.093,0.349,True,False,1990.385,A,3.901,0.048,0.181,False
Item_8,B,-0.298,-0.380,-0.350,-0.052,0.110,-0.476,0.634,0.792,True,False,1993.031,A,0.342,0.559,0.720,False
Item_9,B,1.066,0.998,1.028,-0.037,0.108,-0.346,0.729,0.841,True,False,1995.985,A,0.276,0.599,0.720,False


In [7]:
round(pcm.threshold_dif_table, 3)   # per-item category-width (DCF) test

Group  Reference  Focal  Difference     SE      t        df  \
Item    Category                                                               
Item_1  1            B      4.162  3.902      -0.260  0.323 -0.804  1854.415   
        2            B      2.911  2.989       0.078  0.247  0.317  1876.871   
Item_2  1            B      2.002  1.408      -0.594  0.275 -2.156  1845.923   
        2            B      2.017  2.359       0.343  0.208  1.648  1993.333   
Item_3  1            B      1.337  1.444       0.107  0.211  0.509  1989.472   
        2            B      3.758  3.465      -0.293  0.307 -0.953  1966.154   
Item_4  1            B      1.520  2.504       0.984  0.287  3.431  1993.488   
        2            B      3.469  3.466      -0.003  0.214 -0.016  1994.677   
Item_5  1            B      2.347  2.574       0.227  0.268  0.847  1992.681   
        2            B      3.957  3.746      -0.211  0.247 -0.854  1994.164   
Item_6  1            B      1.139  1.667       0.528  0.208  2.535  1995.143   
        2            B      1.877  1.416      -0.461  0.272 -1.691  1989.466   
Item_7  1            B      1.733  1.325      -0.407  0.249 -1.637  1985.696   
        2            B      3.191  3.248       0.057  0.217  0.261  1994.019   
Item_8  1            B      3.729  3.718      -0.011  0.290 -0.038  1995.620   
        2            B      3.192  3.005      -0.187  0.237 -0.787  1988.991   
Item_9  1            B      1.774  1.832       0.058  0.203  0.288  1995.650   
        2            B      3.253  3.511       0.258  0.313  0.825  1995.997   
Item_10 1            B      3.108  2.939      -0.169  0.291 -0.582  1994.363   
        2            B      1.988  2.156       0.168  0.203  0.830  1994.383   
Item_11 1            B      0.562  0.985       0.422  0.229  1.843  1995.892   
        2            B      1.112  0.696      -0.416  0.279 -1.490  1994.549   
Item_12 1            B      2.977  2.740      -0.236  0.230 -1.026  1992.257   
        2            B      0.828  0.596      -0.233  0.225 -1.032  1993.144   
Item_13 1            B      2.553  2.134      -0.419  0.281 -1.489  1993.548   
        2            B      3.086  2.909      -0.177  0.212 -0.836  1995.952   
Item_14 1            B      2.623  2.483      -0.139  0.224 -0.622  1995.858   
        2            B      2.550  2.480      -0.070  0.230 -0.303  1992.269   
Item_15 1            B      0.895  0.780      -0.115  0.238 -0.484  1993.633   
        2            B      0.466  0.468       0.003  0.248  0.010  1993.824   

                      p  p (corrected)  Flagged  
Item    Category                                 
Item_1  1         0.421          0.751    False  
        2         0.751          0.751    False  
Item_2  1         0.031          0.062    False  
        2         0.099          0.099    False  
Item_3  1         0.611          0.611    False  
        2         0.341          0.611    False  
Item_4  1         0.001          0.001     True  
        2         0.987          0.987    False  
Item_5  1         0.397          0.397    False  
        2         0.393          0.397    False  
Item_6  1         0.011          0.023     True  
        2         0.091          0.091    False  
Item_7  1         0.102          0.203    False  
        2         0.794          0.794    False  
Item_8  1         0.970          0.970    False  
        2         0.431          0.863    False  
Item_9  1         0.773          0.773    False  
        2         0.409          0.773    False  
Item_10 1         0.561          0.561    False  
        2         0.407          0.561    False  
Item_11 1         0.066          0.131    False  
        2         0.136          0.136    False  
Item_12 1         0.305          0.305    False  
        2         0.302          0.305    False  
Item_13 1         0.137          0.273    False  
        2         0.403          0.403    False  
Item_14 1         0.534          0.762    False  
      

Item_4 should be flagged in `threshold_dif_table` (its category structure differs between groups) but not in `dif_table` (its item location is unaffected) — the two phenomena are deliberately kept independent in this simulation so the signal is unambiguous.

#### RSM

Simulate two groups on 20 items sharing one rating scale with 4 categories (maximum score 3). Group B again gets the Item_1/2/3 location DIF, plus the same +0.8 shift to one shared category width used for PCM's Item_4 — since RSM's thresholds (and therefore widths) are shared across all items, this is a uniform instrument-wide DCF effect rather than a single-item one.

In [8]:
no_of_persons = 1000
sim_a = RSM_Sim(no_of_items=20, no_of_persons=no_of_persons, max_score=3, item_range=2, seed=1)

items_b = sim_a.items.copy()
items_b['Item_1'] += 1.0
items_b['Item_2'] -= 1.0
items_b['Item_3'] += 0.5

thresholds_b = sim_a.thresholds.copy()
thresholds_b[1] -= 0.8
thresholds_b -= thresholds_b.mean()

sim_b = RSM_Sim(no_of_items=20, no_of_persons=no_of_persons, max_score=3, item_range=2,
                manual_items=items_b, manual_item_names=sim_a.item_names,
                manual_thresholds=thresholds_b, seed=2)

resp_a, resp_b = sim_a.responses.copy(), sim_b.responses.copy()
resp_a.index = [f'A_{i}' for i in range(no_of_persons)]
resp_b.index = [f'B_{i}' for i in range(no_of_persons)]
responses = pd.concat([resp_a, resp_b])
exogenous = pd.DataFrame({'Group': ['A'] * no_of_persons + ['B'] * no_of_persons}, index=responses.index)

rsm = RSM(responses, exogenous=exogenous)

In [9]:
rsm.dif_test('Group',
             reference='A',
             selection_method='robust_z',
             test='both',
             omnibus=True,
             category=True,
             correction='bh',
             no_of_samples=300)

round(rsm.dif_table, 3)   # item-location DIF

,Group,Reference,Focal,Focal (purified),Difference,SE,t,p,p (corrected),Selected,Flagged,df,Category,LR,p_LR,p_LR (corrected),Flagged_LR
Item,,,,,,,,,,,,,,,,,
Item_1,B,-0.908,-0.048,-0.032,0.876,0.081,10.785,0.000,0.000,False,True,1991.785,C+,126.300,0.000,0.000,True
Item_2,B,-0.381,-1.371,-1.354,-0.974,0.078,-12.460,0.000,0.000,False,True,1995.199,C-,157.531,0.000,0.000,True
Item_3,B,0.665,1.078,1.095,0.430,0.092,4.659,0.000,0.000,False,False,1940.285,A,22.487,0.000,0.000,False
Item_4,B,-0.700,-0.741,-0.726,-0.025,0.080,-0.316,0.752,0.976,True,False,1995.096,A,0.149,0.700,0.895,False
Item_5,B,-0.287,-0.171,-0.146,0.141,0.080,1.764,0.078,0.260,True,False,1997.476,A,3.172,0.075,0.250,False
Item_6,B,0.791,0.780,0.796,0.006,0.092,0.064,0.949,0.976,True,False,1981.289,A,0.004,0.950,0.950,False
Item_7,B,-0.313,-0.374,-0.360,-0.047,0.080,-0.593,0.553,0.920,True,False,1981.294,A,0.315,0.574,0.895,False
Item_8,B,-0.489,-0.545,-0.531,-0.042,0.080,-0.528,0.598,0.920,True,False,1986.245,A,0.051,0.821,0.950,False
Item_9,B,0.967,0.933,0.948,-0.019,0.095,-0.196,0.845,0.976,True,False,1965.983,A,0.210,0.647,0.895,False


In [10]:
rsm.dif_omnibus_table     # joint item-location test

,LR,df,p,Flagged
B,301.957358,19,0.0,True


In [11]:
round(rsm.threshold_dif_table, 3)   # shared category-width (DCF) DIF

,Group,Category,Reference,Focal,Difference,SE,t,p,p (corrected),Flagged,df
0,B,1,3.887,4.788,0.901,0.077,11.763,0.000,0.000,True,1951.796
1,B,2,1.555,1.508,-0.048,0.065,-0.738,0.461,0.461,False,1996.232


#### MFRM

Simulate two groups of 300 persons each, rated by the same 6 raters on 10 items (maximum score 4), under the 'global' rater model. Group B has the same item-location DIF injected on Items 1-3 as the other model classes, plus a genuine category-width (DCF) shift on the shared threshold vector -- raters are held fixed between groups (same rater effects, fresh simulated ratings):

In [12]:
no_of_persons = 300
sim_a = MFRM_Sim_Global(no_of_items=10, no_of_persons=no_of_persons, no_of_facet_elements=6, max_score=4,
                         manual_person_names=[f'A_{i}' for i in range(no_of_persons)], seed=1)

items_b = sim_a.items.copy()
items_b['Item_1'] += 1.0
items_b['Item_2'] -= 1.0
items_b['Item_3'] += 0.5

thresholds_b = sim_a.thresholds.copy()
thresholds_b[1] -= 0.8
thresholds_b -= thresholds_b.mean()

sim_b = MFRM_Sim_Global(no_of_items=10, no_of_persons=no_of_persons, no_of_facet_elements=6, max_score=4,
                         manual_items=items_b, manual_item_names=sim_a.item_names,
                         manual_thresholds=thresholds_b.values,
                         manual_facet_effects=sim_a.facet_effects, manual_facet_names=sim_a.facet_names,
                         manual_person_names=[f'B_{i}' for i in range(no_of_persons)], seed=2)

responses = pd.concat([sim_a.responses, sim_b.responses])
exogenous = pd.DataFrame({'Group': ['A'] * no_of_persons + ['B'] * no_of_persons},
                          index=[f'A_{i}' for i in range(no_of_persons)] + [f'B_{i}' for i in range(no_of_persons)])

mfrm = MFRM(responses, max_score=4, exogenous=exogenous)

`model=` selects which of the 8 rater representations each group is calibrated under -- item-location DIF and DCF are conceptually model-independent, so this only affects each group's internal `calibrate()` call. Run the DIF/DCF test, comparing group B against reference group A:

In [13]:
mfrm.dif_test('Group',
              model='global',
              reference='A',
              selection_method='wald',
              test='both',
              omnibus=True,
              category=True,
              correction='bh',
              welch=True,
              no_of_samples=300,
              seed=1)

round(mfrm.dif_table, 3)   # item-location DIF

Bootstrap resamples: 1/300 resamples | elapsed 0s | about 21s left

Bootstrap resamples: 300/300 resamples | elapsed 4s | about 0s left | done

,Group,Reference,Focal,Focal (purified),Difference,SE,t,p,p (corrected),Selected,Flagged,df,Category,LR,p_LR,p_LR (corrected),Flagged_LR
Item,,,,,,,,,,,,,,,,,
Item_1,B,0.152,1.115,1.157,1.005,0.070,14.453,0.000,0.000,False,True,586.124,C+,265.663,0.000,0.000,True
Item_2,B,1.009,-0.052,-0.010,-1.019,0.057,-17.839,0.000,0.000,False,True,595.669,C-,283.597,0.000,0.000,True
Item_3,B,-0.082,0.306,0.348,0.430,0.065,6.625,0.000,0.000,False,False,594.851,A,51.450,0.000,0.000,False
Item_4,B,0.657,0.726,0.785,0.128,0.065,1.954,0.051,0.128,True,False,597.582,A,4.583,0.032,0.081,False
Item_5,B,-1.048,-1.041,-0.969,0.078,0.060,1.297,0.195,0.325,True,False,596.396,A,1.912,0.167,0.278,False
Item_6,B,-0.284,-0.334,-0.294,-0.010,0.060,-0.170,0.865,0.904,True,False,597.977,A,0.025,0.876,0.876,False
Item_7,B,0.083,0.034,0.075,-0.008,0.067,-0.121,0.904,0.904,True,False,589.089,A,0.029,0.864,0.876,False
Item_8,B,0.256,0.127,0.156,-0.100,0.066,-1.525,0.128,0.256,True,False,595.889,A,2.995,0.084,0.167,False
Item_9,B,-0.530,-0.600,-0.563,-0.033,0.063,-0.518,0.604,0.825,True,False,597.904,A,0.263,0.608,0.837,False


In [14]:
mfrm.dif_omnibus_table   # item-isolated omnibus test (facet_effects pinned at the pooled value)

,LR,df,p,Flagged
B,610.093181,9,0.0,True


In [15]:
round(mfrm.threshold_dif_table, 3)   # DCF -- shared category-width shift, same rationale as RSM's own DCF

,Group,Category,Reference,Focal,Difference,SE,t,p,p (corrected),Flagged,df
0,B,1,3.797,4.547,0.750,0.115,6.491,0.000,0.00,True,596.733
1,B,2,2.544,2.595,0.052,0.058,0.897,0.370,0.37,False,596.056
2,B,3,2.437,2.325,-0.112,0.107,-1.054,0.292,0.37,False,597.587


Items 1-3, where item-location DIF was injected, should be flagged with `Difference` estimates close to the specified shifts. In `threshold_dif_table`, Category 1 (the width between thresholds 1 and 2) should be flagged, where the DCF shift was injected.

Item_3 is a useful edge case: its injected DIF (0.5 logits) is only marginally above the ETS Category B threshold of 0.43. In this simulated sample, data noise pulls the observed `Difference` down to 0.406 – decisively significant (`p` = 0.000) but just under the magnitude gate, so it is correctly *not* flagged. Raising the injected value to 0.55 logits recovers an observed `Difference` of 0.439, just clearing the threshold, and Item_3 is then flagged cleanly (Category 'B+').

`drf_test()` is the DIF analogue for raters: does a rater score the two groups differently? Each group is fitted jointly, and each rater common to both groups is tested in its representation's terms, against the average of the other raters (a pool from which raters that themselves differ between the groups are dropped):

- global effect, `drf_table` (1 df);
- item effects, `drf_item_table` (one fewer df than items; for `'pseudo_halo'` and `'bistretch'`, the change in the item stretch, 1 df);
- threshold shape, `drf_threshold_table` (one fewer df than thresholds; for `'centrality'` and `'bistretch'`, the change in the threshold stretch, 1 df);
- the interaction, `drf_interaction_table` (`'matrix'` only);
- an overall test of all of a rater's components, `drf_overall_table`, and an omnibus test of all raters together, `drf_omnibus_table`.

The p-values are Monte Carlo ones by default (`p_method='monte-carlo'`, `no_of_samples=500`), so that choosing the pool from the same data is part of the reference distribution. As for DIF, a rater is flagged only if the difference is significant after correction and its size reaches `logit_threshold` (0.43 by default). Three examples follow: `'global'`, `'items'` and `'centrality'`.

**`'global'`**: a single global effect per rater. Simulate a fresh pair of groups with the items held fixed between groups, and a shift in global effect of 1.2 logits on Rater_1 in group B:

In [16]:
sim_a = MFRM_Sim_Global(no_of_items=10, no_of_persons=no_of_persons, no_of_facet_elements=6, max_score=4,
                         manual_person_names=[f'A_{i}' for i in range(no_of_persons)], seed=1)

facets_b = sim_a.facet_effects.copy()
facets_b['Rater_1'] += 1.2

sim_b = MFRM_Sim_Global(no_of_items=10, no_of_persons=no_of_persons, no_of_facet_elements=6, max_score=4,
                         manual_items=sim_a.items, manual_item_names=sim_a.item_names,
                         manual_facet_effects=facets_b, manual_facet_names=sim_a.facet_names,
                         manual_person_names=[f'B_{i}' for i in range(no_of_persons)], seed=2)

responses = pd.concat([sim_a.responses, sim_b.responses])
mfrm = MFRM(responses, max_score=4, exogenous=exogenous)

mfrm.drf_test('Group',
              model='global',
              reference='A',
              seed=1)

round(mfrm.drf_table, 3)

,Group,Representation,Reference,Focal,Difference,SE,T,df,Denominator df,p,p (corrected),Flagged,Pool,Cells left out,Note
Rater,,,,,,,,,,,,,,,
Rater_1,B,global,-0.453,0.571,1.255,0.050,619.279,1.0,590.393,0.002,0.012,True,4,0,
Rater_2,B,global,-0.027,-0.293,-0.048,0.051,0.894,1.0,592.074,0.401,0.481,False,3,0,
Rater_3,B,global,-0.158,-0.428,-0.052,0.050,1.058,1.0,586.081,0.369,0.481,False,3,0,
Rater_4,B,global,1.245,1.074,0.080,0.053,2.287,1.0,590.569,0.170,0.339,False,3,0,
Rater_5,B,global,-0.760,-0.860,0.131,0.047,7.806,1.0,597.807,0.014,0.042,False,4,0,
Rater_6,B,global,0.153,-0.063,0.020,0.048,0.167,1.0,597.929,0.723,0.723,False,3,0,


Rater_1 is flagged: its global effect is 1.255 logits higher in group B (SE 0.050; 1.2 injected). Its p = 0.002 is the smallest a Monte Carlo p-value from 500 draws can be. Rater_5 is also significant after correction (p = 0.006), but its difference, 0.131 logits, is below `logit_threshold`, so it is not flagged. The omnibus test asks whether any rater differs between the groups:

In [17]:
mfrm.drf_omnibus_table

,T,df,F,Denominator df,p,Flagged
B,641.092518,5,127.352226,588.043818,0.0,True


T = 641.1 on 5 df, referred to an F distribution (F = 127.4 on 5 and 594 df): at least one rater differs.

**`'items'`**: each rater has its own effect on each item. Shift Rater_1's effect on Item_1 by 1.2 logits in group B, with every other effect, and every item location, held fixed:

In [18]:
sim_a = MFRM_Sim_Items(no_of_items=10, no_of_persons=no_of_persons, no_of_facet_elements=6, max_score=4,
                        manual_person_names=[f'A_{i}' for i in range(no_of_persons)], seed=1)

facets_b = sim_a.facet_effects.copy()   # (rater x item) DataFrame
facets_b.loc['Rater_1', 'Item_1'] += 1.2

sim_b = MFRM_Sim_Items(no_of_items=10, no_of_persons=no_of_persons, no_of_facet_elements=6, max_score=4,
                        manual_items=sim_a.items, manual_item_names=sim_a.item_names,
                        manual_facet_effects=facets_b, manual_facet_names=sim_a.facet_names,
                        manual_person_names=[f'B_{i}' for i in range(no_of_persons)], seed=2)

responses = pd.concat([sim_a.responses, sim_b.responses])
mfrm = MFRM(responses, max_score=4, exogenous=exogenous)

mfrm.drf_test('Group',
              model='items',
              reference='A',
              seed=1)

round(mfrm.drf_item_table[['T', 'df', 'p (corrected)', 'Flagged']], 3)

,T,df,p (corrected),Flagged
Rater,,,,
Rater_1,54.707,9.0,0.012,True
Rater_2,15.593,9.0,0.257,False
Rater_3,12.275,9.0,0.320,False
Rater_4,12.326,9.0,0.326,False
Rater_5,13.487,9.0,0.319,False
Rater_6,8.401,9.0,0.549,False


Rater_1's item effects differ between the groups (T = 54.7 on 9 df, corrected p = 0.012); no other rater's do. A shift on one item barely moves a rater's global effect, so `drf_table` (not shown) does not flag Rater_1. The item test's breakdown shows where the difference is: each item's difference (Rater_1's level plus its effect on that item), with Holm-corrected p-values:

In [19]:
rater_1 = mfrm.drf_item_table.loc['Rater_1']
items = sim_a.item_names
pd.DataFrame({'Difference': [rater_1[f'Difference {i}'] for i in items],
              'SE': [rater_1[f'SE {i}'] for i in items],
              'p (Holm)': [rater_1[f'p {i} (Holm)'] for i in items]},
             index=items).round(3)

,Difference,SE,p (Holm)
Item_1,1.122,0.148,0.0
Item_2,-0.095,0.147,1.0
Item_3,-0.145,0.145,1.0
Item_4,0.082,0.150,1.0
Item_5,0.170,0.143,1.0
Item_6,0.109,0.140,1.0
Item_7,0.126,0.140,1.0
Item_8,-0.118,0.147,1.0
Item_9,0.168,0.153,1.0
Item_10,-0.035,0.144,1.0


Only Item_1 differs: by 1.122 logits (SE 0.148; 1.2 injected).

**`'centrality'`**: each rater has a global effect ($\lambda_r$) and a threshold stretch ($\omega_r$, above 1 for a rater who avoids the extreme categories, below 1 for one who favours them). In group B, shift Rater_1's global effect by 1.2 logits, and stretch Rater_2's thresholds by a factor of 1.6, with everything else unchanged:

In [20]:
sim_a = MFRM_Sim_Centrality(no_of_items=10, no_of_persons=no_of_persons, no_of_facet_elements=6, max_score=4,
                             manual_person_names=[f'A_{i}' for i in range(no_of_persons)], seed=1)

lambda_b = sim_a.lambda_.values.copy()
lambda_b[0] += 1.2   # Rater_1
omega_b = sim_a.omega.values.copy()
omega_b[1] *= 1.6    # Rater_2

sim_b = MFRM_Sim_Centrality(no_of_items=10, no_of_persons=no_of_persons, no_of_facet_elements=6, max_score=4,
                             manual_items=sim_a.items, manual_item_names=sim_a.item_names,
                             manual_thresholds=sim_a.thresholds.values,
                             manual_lambda=lambda_b, manual_omega=omega_b,
                             manual_person_names=[f'B_{i}' for i in range(no_of_persons)], seed=2)

responses = pd.concat([sim_a.responses, sim_b.responses])
mfrm = MFRM(responses, max_score=4, exogenous=exogenous)

mfrm.drf_test('Group',
              model='centrality',
              reference='A',
              seed=1)

pd.concat({'Global effect': mfrm.drf_table[['Difference', 'SE', 'p (corrected)', 'Flagged']],
           'Threshold stretch': mfrm.drf_threshold_table[['T', 'p (corrected)', 'Flagged']]}, axis=1).round(3)

Global effect                              Threshold stretch  \
           Difference     SE p (corrected) Flagged                 T   
Rater                                                                  
Rater_1         1.311  0.068         0.012    True             0.026   
Rater_2        -0.052  0.069         0.575   False           178.826   
Rater_3        -0.057  0.066         0.572   False             1.519   
Rater_4         0.095  0.062         0.303   False             0.718   
Rater_5         0.176  0.064         0.018   False             1.643   
Rater_6         0.014  0.059         0.840   False             1.499   

                               
        p (corrected) Flagged  
Rater                          
Rater_1         0.854   False  
Rater_2         0.012    True  
Rater_3         0.353   False  
Rater_4         0.501   False  
Rater_5         0.353   False  
Rater_6         0.353   False

Each change is found by its own test. Rater_1's global effect differs (1.311 logits, SE 0.068; 1.2 injected) but its threshold stretch does not (corrected p = 0.842); Rater_2's threshold stretch differs (T = 178.8 on 1 df, corrected p = 0.012) but its global effect does not (corrected p = 0.596). Rater_5's global effect is significant after correction (p = 0.024), but at 0.176 logits it is below `logit_threshold`.

#### Groups scored by different raters, or responding to different items


Groups are often scored by partly overlapping sets of raters, or respond to partly overlapping sets of items. `dif_test()` handles both: each group is calibrated on the items its persons responded to, only items shared by both groups are compared, and, for `MFRM`, each group is anchored on the facet elements shared by both groups before item locations and thresholds are compared.

**`MFRM`, partly overlapping raters**: group A is rated by Rater_1-6 and group B by Rater_3-8, under the `'items'` model (each rater has its own effect on each item). DIF is injected on Item_1 only:


In [21]:
sim_a = MFRM_Sim_Items(no_of_items=10, no_of_persons=no_of_persons, no_of_facet_elements=8, max_score=4,
                        manual_person_names=[f'A_{i}' for i in range(no_of_persons)], seed=1)

items_b = sim_a.items.copy()
items_b['Item_1'] += 1.0

sim_b = MFRM_Sim_Items(no_of_items=10, no_of_persons=no_of_persons, no_of_facet_elements=8, max_score=4,
                        manual_items=items_b, manual_item_names=sim_a.item_names,
                        manual_facet_effects=sim_a.facet_effects, manual_facet_names=sim_a.facet_names,
                        manual_person_names=[f'B_{i}' for i in range(no_of_persons)], seed=2)

sim_a.drop_raters(['Rater_7', 'Rater_8'])   # group A: Rater_1-6
sim_b.drop_raters(['Rater_1', 'Rater_2'])   # group B: Rater_3-8

responses = pd.concat([sim_a.responses, sim_b.responses])
persons = responses.index.get_level_values(1).unique()
exogenous = pd.DataFrame({'Group': [p[0] for p in persons]}, index=persons)

mfrm = MFRM(responses, max_score=4, exogenous=exogenous)
mfrm.dif_test('Group',
              model='items',
              reference='A',
              no_of_samples=300,
              seed=1)

round(mfrm.dif_table, 3)

/var/folders/hs/f4mv8_3d65bf5fln8jlxhxpc0000gn/T/ipykernel_8968/1362870342.py:12: UserWarning: 2 facet elements (e.g. raters) removed: ['Rater_7', 'Rater_8'].
  sim_a.drop_raters(['Rater_7', 'Rater_8'])   # group A: Rater_1-6
/var/folders/hs/f4mv8_3d65bf5fln8jlxhxpc0000gn/T/ipykernel_8968/1362870342.py:13: UserWarning: 2 facet elements (e.g. raters) removed: ['Rater_1', 'Rater_2'].
  sim_b.drop_raters(['Rater_1', 'Rater_2'])   # group B: Rater_3-8


,Group,Reference,Focal,Focal (purified),Difference,SE,z,p,p (corrected),Selected,Flagged
Item,,,,,,,,,,,
Item_1,B,0.355,1.254,1.350,0.995,0.079,12.635,0.000,0.000,False,True
Item_2,B,1.004,0.964,1.068,0.064,0.077,0.832,0.406,0.563,True,False
Item_3,B,-0.206,-0.239,-0.133,0.073,0.066,1.108,0.268,0.563,True,False
Item_4,B,0.437,0.274,0.362,-0.075,0.069,-1.096,0.273,0.563,True,False
Item_5,B,-1.148,-1.227,-1.128,0.020,0.075,0.262,0.794,0.794,True,False
Item_6,B,-0.067,-0.278,-0.194,-0.126,0.083,-1.513,0.130,0.563,True,False
Item_7,B,0.147,0.100,0.203,0.056,0.074,0.755,0.450,0.563,True,False
Item_8,B,-0.016,-0.128,-0.034,-0.018,0.067,-0.269,0.788,0.794,True,False
Item_9,B,-0.234,-0.405,-0.316,-0.082,0.082,-1.004,0.315,0.563,True,False


Only Item_1 should be flagged. Each group's item locations are averages over the raters that scored it, so without anchoring on the raters shared by both groups (Rater_3-6), the differences between the two rater sets would show up as apparent DIF on the other items.

**`SLM`, partly overlapping items**: group A responded to Items 1-10 and group B to Items 3-12, with DIF injected on Item_5:


In [22]:
sim_a = SLM_Sim(no_of_items=12, no_of_persons=no_of_persons, item_range=2, seed=1)

items_b = sim_a.items.copy()
items_b['Item_5'] += 1.0

sim_b = SLM_Sim(no_of_items=12, no_of_persons=no_of_persons, item_range=2,
                manual_items=items_b, manual_item_names=sim_a.item_names, seed=2)

resp_a, resp_b = sim_a.responses.copy(), sim_b.responses.copy()
resp_a.index = [f'A_{i}' for i in range(no_of_persons)]
resp_b.index = [f'B_{i}' for i in range(no_of_persons)]
resp_a[['Item_11', 'Item_12']] = np.nan   # group A: Items 1-10
resp_b[['Item_1', 'Item_2']] = np.nan     # group B: Items 3-12
responses = pd.concat([resp_a, resp_b])

exogenous = pd.DataFrame({'Group': ['A'] * no_of_persons + ['B'] * no_of_persons}, index=responses.index)

slm = SLM(responses, exogenous=exogenous)
slm.dif_test('Group',
             reference='A')

round(slm.dif_table, 3)

,Group,Reference,Focal,Focal (purified),Difference,SE,t,p,p (corrected),Selected,Flagged,df
Item,,,,,,,,,,,,
Item_3,B,-0.080,-0.030,0.086,0.166,0.217,0.763,0.446,0.869,True,False,525.805
Item_4,B,0.717,0.352,0.400,-0.317,0.222,-1.429,0.154,0.615,True,False,526.081
Item_5,B,-1.106,-0.227,-0.135,0.972,0.225,4.326,0.000,0.000,False,True,523.861
Item_6,B,-0.184,-0.312,-0.226,-0.041,0.217,-0.191,0.849,0.869,True,False,526.950
Item_7,B,0.205,0.143,0.241,0.036,0.218,0.165,0.869,0.869,True,False,526.784
Item_8,B,0.096,-0.030,0.057,-0.039,0.221,-0.178,0.859,0.869,True,False,526.764
Item_9,B,-0.721,-0.722,-0.615,0.106,0.225,0.471,0.638,0.869,True,False,526.990
Item_10,B,-0.279,-0.300,-0.195,0.084,0.227,0.372,0.710,0.869,True,False,526.931


Only the items shared by both groups (Items 3-10) appear in `dif_table`, and only Item_5 should be flagged. With more than two groups, a focal group that shares no items with the reference group cannot be compared with it; it is left out with a `UserWarning` and recorded in `slm.dif_excluded_groups`.
